# 4. Audit and analyze retail orders
## Goal
Load a CSV, define valid data, reconcile row counts, validate a join, and summarize revenue.
Course Module 6. This is invented teaching data, not a real retailer.
## Setup
Requires the included `data/orders.csv`. Open the course folder or run from its `notebooks/` folder. See `data/README.md` for provenance and the data dictionary.
Reference: [pandas tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html).
## Steps
### 1. Load raw data without editing it

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates = [Path("../data/orders.csv"), Path("data/orders.csv"), Path("orders.csv")]
source = next((path for path in candidates if path.exists()), None)
if source is None:
    raise FileNotFoundError("Put the course data/orders.csv beside the course notebooks folder.")
raw = pd.read_csv(source)
print("Raw shape:", raw.shape)
print(raw.head().to_string(index=False))
print("Missing cells by column:")
print(raw.isna().sum().to_string())

Raw shape: (62, 6)
order_id       date customer_id region units  unit_price
    O000 2026-01-01         C00  North     1        10.0
    O001 2026-01-01         C01  South     2        20.0
    O002 2026-01-01         C02   East     3        30.0
    O003 2026-01-01         C03  North     4        15.0
    O004 2026-01-02         C04  South    -1        25.0
Missing cells by column:
order_id       0
date           0
customer_id    0
region         1
units          0
unit_price     1


### 2. State policy, then clean
Grain: one order with one product. For this synthetic dataset, repeated order IDs are known exact duplicate records: remove exact duplicates first, then reject remaining duplicate IDs as ambiguous. Normalize region whitespace/case and label missing region Unknown. Reject invalid dates, nonfinite/nonpositive/noninteger units, and missing/nonfinite/negative prices. Zero price is allowed. A rejected row counts once even if it has multiple reasons.

In [2]:
unique = raw.drop_duplicates().copy()
duplicate_rows = len(raw) - len(unique)
if not unique["order_id"].is_unique:
    raise ValueError("Conflicting rows share an order ID; resolve before analyzing.")
unique["date"] = pd.to_datetime(unique["date"], format="%Y-%m-%d", errors="coerce")
unique["units"] = pd.to_numeric(unique["units"], errors="coerce")
unique["unit_price"] = pd.to_numeric(unique["unit_price"], errors="coerce")
unique["region"] = unique["region"].astype("string").str.strip().str.title().replace("", pd.NA).fillna("Unknown")
bad_date = unique["date"].isna()
bad_units = (~np.isfinite(unique["units"])) | (unique["units"] <= 0) | (unique["units"] % 1 != 0)
bad_price = (~np.isfinite(unique["unit_price"])) | (unique["unit_price"] < 0)
reasons = pd.DataFrame({"invalid_date": bad_date, "invalid_units": bad_units, "invalid_price": bad_price})
invalid = reasons.any(axis=1)
rejected = unique.loc[invalid].copy()
rejected["reason"] = reasons.loc[invalid].apply(lambda row: ", ".join(row.index[row]), axis=1)
clean = unique.loc[~invalid].copy()
clean["revenue"] = clean["units"] * clean["unit_price"]
print(rejected[["order_id", "reason"]].to_string(index=False))

order_id        reason
    O004 invalid_units
    O007 invalid_price
    O010  invalid_date
    O013 invalid_units


### 3. Reconcile before calculating
These checks test row identity, exclusions, and valid values.

In [3]:
audit = {"raw": len(raw), "exact_duplicates": duplicate_rows, "rejected": len(rejected), "clean": len(clean)}
print(audit)
assert audit == {"raw": 62, "exact_duplicates": 2, "rejected": 4, "clean": 56}
assert len(raw) == duplicate_rows + len(rejected) + len(clean)
assert clean["order_id"].is_unique
assert clean["revenue"].ge(0).all()
assert clean["date"].notna().all()

{'raw': 62, 'exact_duplicates': 2, 'rejected': 4, 'clean': 56}


### 4. Validate a dimension join
The customer table must have one row per customer; otherwise the join can multiply revenue.

In [4]:
customers = pd.DataFrame({"customer_id": [f"C{i:02d}" for i in range(12)], "tier": ["Standard", "Member"] * 6})
joined = clean.merge(customers, on="customer_id", how="left", validate="many_to_one", indicator=True)
assert len(joined) == len(clean)
assert joined["_merge"].eq("both").all()
assert np.isclose(joined["revenue"].sum(), clean["revenue"].sum())
print("All customer keys matched without multiplying rows.")

All customer keys matched without multiplying rows.


### 5. Summarize with counts and units
Unknown is retained as an explicit region group; excluding it silently would change totals.

In [5]:
summary = clean.groupby("region", as_index=False).agg(orders=("order_id", "size"), revenue_aed=("revenue", "sum"), average_order_aed=("revenue", "mean"))
print(summary.to_string(index=False))
overall_average = clean["revenue"].mean()
weighted_average = np.average(summary["average_order_aed"], weights=summary["orders"])
assert np.isclose(overall_average, weighted_average)
assert np.isclose(summary["revenue_aed"].sum(), clean["revenue"].sum())
print(f"Overall average order value: {overall_average:.2f} AED")

 region  orders  revenue_aed  average_order_aed
   East      20       1000.0          50.000000
  North      20       1000.0          50.000000
  South      15        775.0          51.666667
Unknown       1         20.0          20.000000
Overall average order value: 49.91 AED


### 6. Save derived outputs separately
The raw CSV is preserved. These files are small and are recreated on rerun.

In [6]:
output_dir = source.resolve().parent.parent / "outputs"
output_dir.mkdir(exist_ok=True)
clean.to_csv(output_dir / "orders_clean.csv", index=False)
rejected.to_csv(output_dir / "orders_rejected.csv", index=False)
summary.to_csv(output_dir / "region_summary.csv", index=False)
print("Saved clean, rejected, and summary CSV files in outputs/.")

Saved clean, rejected, and summary CSV files in outputs/.


## Checks
The reconciliation is 62 = 2 + 4 + 56. Explain why missing region can remain Unknown while missing price is rejected under this policy. These choices belong to this dataset's contract, not every business dataset.
## Next Steps
Complete exercises 19–22. Write three findings using actual outputs. Add a conflicting duplicate order to a copy of the source and make your pipeline stop; do not overwrite the original course data.